# Network evolution — starter analysis

Loads the newest network snapshot from `data/research_private/snapshots/`
and shows the co-mention graph's macro structure + Louvain community
membership. Definitions live in `research/methodology.md` §3; graph
construction is in `pipeline/research/network_metrics.py`.

To track *evolution* rather than a single day, extend the last cell
to iterate over `SNAPSHOTS.iterdir()` and compare `network_metrics.json`
across dates.

In [ ]:
import json
from pathlib import Path

import networkx as nx
import pandas as pd

REPO_ROOT = Path.cwd().parents[1]
SNAPSHOTS = REPO_ROOT / "data" / "research_private" / "snapshots"
assert SNAPSHOTS.exists(), (
    f"No snapshots yet. Run `python -m pipeline.research.snapshot` first.\n"
    f"Expected: {SNAPSHOTS}"
)

latest = sorted(p for p in SNAPSHOTS.iterdir() if p.is_dir())[-1]
print(f"Latest snapshot: {latest.name}")

In [ ]:
edges = pd.read_parquet(latest / "network_edges.parquet")
nodes = pd.read_parquet(latest / "network_nodes.parquet")
communities = pd.read_parquet(latest / "network_communities.parquet")
metrics = json.loads((latest / "network_metrics.json").read_text(encoding="utf-8"))

print("Graph metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print(f"\ncommunities: {communities['community_id'].nunique()}")
nodes.head()

## Top 15 nodes by PageRank

PageRank on the weighted co-mention graph proxies each entity's
structural centrality — a good candidate list for 'stable hubs'
in the paper's structural chapter.

In [ ]:
nodes.nlargest(15, "pagerank")

## Louvain community sizes

In [ ]:
sizes = communities.groupby("community_id").size().sort_values(ascending=False)
sizes.head(10)

## Community roster (first three communities)

In [ ]:
for cid in sizes.head(3).index:
    members = communities[communities["community_id"] == cid]["entity"].tolist()
    print(f"Community {cid} ({len(members)} nodes): {', '.join(members[:20])}"
          + (" ..." if len(members) > 20 else ""))

## Rebuild the graph and draw a small preview

Draws only the top-degree subgraph to keep the layout readable — the
full graph is dense enough that a static PNG turns into a hairball.

In [ ]:
g = nx.Graph()
for row in edges.itertuples(index=False):
    g.add_edge(row.source, row.target, weight=int(row.weight))

top_nodes = nodes.nlargest(25, "pagerank")["entity"].tolist()
sub = g.subgraph(top_nodes).copy()
print(f"Preview subgraph: {sub.number_of_nodes()} nodes / {sub.number_of_edges()} edges")

pos = nx.spring_layout(sub, seed=42, k=0.6)
nx.draw_networkx_nodes(sub, pos, node_size=180)
nx.draw_networkx_edges(sub, pos, alpha=0.4, width=[sub[u][v]["weight"] * 0.4 for u, v in sub.edges()])
nx.draw_networkx_labels(sub, pos, font_size=8)

## Evolution — compare current network metrics to earliest snapshot

Once you have at least two snapshots, this cell surfaces the raw Δ
in structural stats. For the paper's evolution chapter you'll want
to widen this to a full time series.

In [ ]:
snapshots = sorted(p for p in SNAPSHOTS.iterdir() if p.is_dir())
if len(snapshots) < 2:
    print("Need at least two snapshots — rerun `python -m pipeline.research.snapshot` on another day.")
else:
    first = snapshots[0]
    last = snapshots[-1]
    first_metrics = json.loads((first / "network_metrics.json").read_text(encoding="utf-8"))
    last_metrics = json.loads((last / "network_metrics.json").read_text(encoding="utf-8"))
    # Numeric keys only — trust_flag (a string, added by P2) would break
    # the Δ subtraction, and pre-P2 snapshots may lack it entirely.
    numeric_keys = [k for k in first_metrics
                    if isinstance(first_metrics[k], (int, float))
                    and isinstance(last_metrics.get(k), (int, float))]
    delta = pd.DataFrame({"first (" + first.name + ")": {k: first_metrics[k] for k in numeric_keys},
                          "last (" + last.name + ")": {k: last_metrics[k] for k in numeric_keys}})
    delta["Δ"] = delta.iloc[:, 1] - delta.iloc[:, 0]
    print(f"trust: first={first_metrics.get('trust_flag', 'unknown')} "
          f"last={last_metrics.get('trust_flag', 'unknown')}")
    delta